# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

In [3]:
import os
import ast
import json
from typing import Literal

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.parsers import PydanticOutputParser
from lib.state_machine import Run
from lib.tooling import tool
from lib.vector_db import VectorStore

In [4]:
# Load environment variables (values are never printed)
load_dotenv(find_dotenv(), override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

MODEL_NAME = "gpt-4o-mini"
print("Keys loaded:", all([OPENAI_API_KEY, OPENAI_BASE_URL, TAVILY_API_KEY]))

Keys loaded: True


In [5]:
# Same path, same collection name and same embedding function as Part 1
chroma_client = chromadb.PersistentClient(path="chromadb")

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)
vector_store = VectorStore(collection)   # reusable wrapper from lib/
print(f"Loaded collection '{collection.name}' with {collection.count()} games.")
assert collection.count() > 0, "Run Part 1 first to build the vector database."

Loaded collection 'udaplay' with 15 games.


### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [6]:
@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, Playstation 5, Xbox 360...)
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    """
    results = vector_store.query(query_texts=[query], n_results=3)
    games = []
    for doc_id, meta, dist in zip(results["ids"][0], results["metadatas"][0], results["distances"][0]):
        games.append({
            "id": doc_id,
            "Name": meta.get("Name"),
            "Platform": meta.get("Platform"),
            "YearOfRelease": meta.get("YearOfRelease"),
            "Genre": meta.get("Genre"),
            "Publisher": meta.get("Publisher"),
            "Description": meta.get("Description"),
            "distance": round(dist, 4),
        })
    return games


# Quick sanity check of the tool
for g in retrieve_game("When was Pokémon Gold and Silver released?"):
    print(g["Name"], "|", g["Platform"], "|", g["YearOfRelease"], "| distance:", g["distance"])

Pokémon Gold and Silver | Game Boy Color | 1999 | distance: 0.108
Pokémon Ruby and Sapphire | Game Boy Advance | 2002 | distance: 0.1444
Super Mario 64 | Nintendo 64 | 1996 | distance: 0.2277


#### Evaluate Retrieval Tool

In [7]:
class EvaluationReport(BaseModel):
    """Structured verdict of the LLM judge."""
    useful: bool = Field(description="Whether the documents are useful to answer the question")
    description: str = Field(description="Detailed explanation of the evaluation result")
    confidence: Literal["high", "medium", "low"] = Field(description="Confidence that the documents fully answer the question")


judge_llm = LLM(model=MODEL_NAME, temperature=0.0)
evaluation_parser = PydanticOutputParser(model_class=EvaluationReport)


@tool
def evaluate_retrieval(question: str, retrieved_docs: str) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - description: description about the evaluation result
    - confidence: high, medium or low
    """
    prompt = (
        "You are an expert judge for a video game knowledge base.\n"
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not.\n"
        "Mark useful=true if the documents mention the game(s) the question is about and contain the facts "
        "needed to answer it. A reasonable inference from the documents counts (for example comparing release "
        "years or descriptions across the retrieved games). "
        "Mark useful=false if the game is not in the documents or a needed fact is missing "
        "(e.g. the asked platform is not listed for that game).\n\n"
        f"# Question:\n{question}\n\n"
        f"# Retrieved documents:\n{retrieved_docs}\n"
    )
    response = judge_llm.invoke(prompt, response_format=EvaluationReport)
    report = evaluation_parser.parse(response)
    return report.model_dump()


# Quick sanity check of the judge
print(evaluate_retrieval("When was Pokémon Gold and Silver released?",
                         json.dumps(retrieve_game("When was Pokémon Gold and Silver released?"))))

{'useful': True, 'description': "The retrieved documents include information specifically about 'Pokémon Gold and Silver', including its release year of 1999. This directly answers the question regarding when Pokémon Gold and Silver was released. The other documents retrieved do not pertain to the question but do not detract from the usefulness of the relevant document. Therefore, the documents are sufficient to answer the query.", 'confidence': 'high'}


#### Game Web Search Tool

In [8]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


@tool
def game_web_search(question: str) -> dict:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - question: a question about game industry.

    Web search fallback (Tavily). Use ONLY when evaluate_retrieval says the internal documents are not useful.
    Returns a short answer plus the title, URL and snippet of each web result, so the sources can be cited.
    """
    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        include_answer=True,
        max_results=5,
    )
    return {
        "answer": response.get("answer"),
        "results": [
            {"title": r.get("title"), "url": r.get("url"), "content": (r.get("content") or "")[:400]}
            for r in response.get("results", [])
        ],
    }

### Agent

In [9]:
INSTRUCTIONS = """
You are UdaPlay, an AI research assistant for the video game industry.

For EVERY user message - including follow-up questions - follow this workflow exactly, calling ONE tool
at a time. Your FIRST action for each new user message MUST be a `retrieve_game` tool call, even if you think
the conversation history already contains the answer. Conversation history is used only to understand what
the user refers to; it is NOT a source. Answering without calling the tools first is an error.
1. Call `retrieve_game` with a standalone search query. For follow-up questions, use the conversation
   history to resolve references such as "it" or "that game" into the game name.
2. Call `evaluate_retrieval` with the user's question and the documents returned by `retrieve_game`
   (pass them as JSON text).
3. Only if the evaluation says useful=false, call `game_web_search` with the question.
   If useful=true, do NOT search the web - answer from the internal documents.
4. Write the final answer.

Rules:
- Use only facts returned by the tools. Never invent release dates, platforms or publishers.
- When the web was used, ALWAYS cite the source URL(s) you relied on.
- When the internal database was used, cite it as "UdaPlay internal knowledge base" with the game ID(s).
- Report the confidence: use the evaluation's confidence for internal answers; for web answers say how well
  the sources agree (high / medium / low).

Final answer format (Markdown):
**Answer:** <one short sentence>
**Details:**
- <key facts: game, platform, year, publisher...>
**Source:** <UdaPlay internal knowledge base (IDs) OR web URL(s)>
**Path:** <internal | web>
**Confidence:** <high | medium | low>
"""

udaplay_agent = Agent(
    model_name=MODEL_NAME,
    instructions=INSTRUCTIONS,
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    temperature=0.0,
)
print("Agent tools:", [t.name for t in udaplay_agent.tools])
print("State machine:", udaplay_agent.workflow)
print("Steps:", list(udaplay_agent.workflow.steps.keys()))

Agent tools: ['retrieve_game', 'evaluate_retrieval', 'game_web_search']
State machine: StateMachine(schema=['user_query', 'instructions', 'messages', 'current_tool_calls', 'total_tokens'])
Steps: ['__entry__', 'message_prep', 'llm_processor', 'tool_executor', '__termination__']


### Trace helper
`show_trace` walks the messages of a run (only the messages produced by the current query) and prints the
state-machine path, every tool call in order (name, arguments, short result), the evaluation verdict,
whether the web fallback was used and the final answer.

In [10]:
def _parse_tool_content(content: str):
    """ToolMessage content is json.dumps(str(result)); turn it back into Python data."""
    try:
        raw = json.loads(content)
    except Exception:
        raw = content
    try:
        return ast.literal_eval(raw)
    except Exception:
        return raw


def _short(text, n=160):
    text = str(text).replace("\n", " ")
    return text if len(text) <= n else text[:n] + "..."


trace_log = []   # collects one summary row per query (used for the report)


def show_trace(run: Run, query: str) -> dict:
    """Print a readable trace of one agent run and return a summary dict."""
    state = run.get_final_state()
    messages = state["messages"]

    # Keep only the messages of THIS query (the session also holds earlier turns)
    start = max(i for i, m in enumerate(messages) if isinstance(m, UserMessage) and m.content == query)
    turn = messages[start:]

    print("=" * 100)
    print(f"QUESTION: {query}")
    print("-" * 100)
    steps = [s.step_id for s in run.snapshots]
    print("State machine path:", " -> ".join(steps + ["__termination__"]))

    verdict, web_used, web_urls, n = None, False, [], 0
    for msg in turn:
        if isinstance(msg, AIMessage) and msg.tool_calls:
            for call in msg.tool_calls:
                n += 1
                args = json.loads(call.function.arguments)
                shown = {k: _short(v, 80) for k, v in args.items()}
                print(f"\n[{n}] TOOL CALL -> {call.function.name}({shown})")
        elif isinstance(msg, ToolMessage):
            result = _parse_tool_content(msg.content)
            if msg.name == "retrieve_game" and isinstance(result, list):
                for g in result:
                    print(f"      result: {g['Name']} [{g['Platform']}, {g['YearOfRelease']}] (id {g['id']}, distance {g['distance']})")
            elif msg.name == "evaluate_retrieval" and isinstance(result, dict):
                verdict = result
                print(f"      VERDICT: useful={result['useful']} | confidence={result['confidence']}")
                print(f"      reason : {_short(result['description'], 300)}")
            elif msg.name == "game_web_search" and isinstance(result, dict):
                web_used = True
                print(f"      web answer: {_short(result.get('answer'), 300)}")
                for r in result.get("results", []):
                    web_urls.append(r["url"])
                    print(f"      source: {_short(r['title'], 70)} - {r['url']}")
            else:
                print(f"      result: {_short(result)}")

    final = turn[-1].content if isinstance(turn[-1], AIMessage) else ""
    print("\n" + "-" * 100)
    print("Evaluation verdict :", "useful (internal data is enough)" if verdict and verdict["useful"]
          else ("NOT useful -> fallback needed" if verdict else "no evaluation call"))
    print("Web fallback used  :", "YES" if web_used else "NO")
    print("Tokens (cumulative session):", state.get("total_tokens"))
    print("\nFINAL ANSWER:\n")
    print(final)
    print("=" * 100)

    summary = {
        "question": query,
        "path": "web" if web_used else "internal",
        "useful": verdict["useful"] if verdict else None,
        "confidence": verdict["confidence"] if verdict else None,
        "web_urls": web_urls,
        "final_answer": final,
    }
    trace_log.append(summary)
    return summary


def ask(question: str, session_id: str = "default") -> None:
    """Invoke the agent in a session and print the trace (the summary is stored in trace_log)."""
    run = udaplay_agent.invoke(question, session_id=session_id)
    show_trace(run, question)

### Run the example queries
Each query uses its own session so the traces are independent. Query 3 (Mortal Kombat X) is **not** in the
local dataset, so the evaluation should reject the internal documents and trigger the web fallback.

In [11]:
# Query 1 - in the dataset (release date)
ask("When was Pokémon Gold and Silver released?", session_id="q1")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: When was Pokémon Gold and Silver released?
----------------------------------------------------------------------------------------------------
State machine path: __entry__ -> message_prep -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor -> __termination__

[1] TOOL CALL -> retrieve_game({'query': 'Pokémon Gold and Silver release date'})
      result: Pokémon Gold and Silver [Game Boy Color, 1999] (id 006, distance 0.1193)
      result: Pokémon Ruby and Sapphire [Game Boy Advance, 2002] (id 007, distance 0.1546)
      result: Super Mario 64 [Nintendo 64, 1996] (id 009, distance 0.2369)

[2] TOOL CALL -> evaluate_retrieval({'question': 'When was Pokémon Gold and Silver released?', 'retrieved_docs': '[{"id":"006","Name":"Pokémon Gold and Silver","Platform":"Game Boy Color","YearO...'})
      VERDICT: useful=True | confidence=high
      reason : The retri

In [12]:
# Query 2 - in the dataset (first 3D platformer Mario game)
ask("Which one was the first 3D platformer Mario game?", session_id="q2")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: Which one was the first 3D platformer Mario game?
----------------------------------------------------------------------------------------------------
State machine path: __entry__ -> message_prep -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor -> __termination__

[1] TOOL CALL -> retrieve_game({'query': 'first 3D platformer Mario game'})
      result: Super Mario 64 [Nintendo 64, 1996] (id 009, distance 0.1079)
      result: Super Mario World [Super Nintendo Entertainment System (SNES), 1990] (id 008, distance 0.1274)
      result: Super Smash Bros. Melee [GameCube, 2001] (id 010, distance 0.1772)

[2] TOOL CALL -> evaluate_retrieval({'question': 'Which one was the first 3D platformer Mario game?', 'retrieved_docs': '[{"id":"009","Name":"Super Mario 64","Platform":"Nintendo 64","YearOfRelease":19...'})
      VERDICT: useful=True | confidence=high
    

In [13]:
# Query 3 - NOT in the dataset -> web fallback
ask("Was Mortal Kombat X released for PlayStation 5?", session_id="q3")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: Was Mortal Kombat X released for PlayStation 5?
----------------------------------------------------------------------------------------------------
State machine path: __entry__ -> message_prep -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor -> __termination__

[1] TOOL CALL -> retrieve_game({'query': 'Mortal Kombat X PlayStation 5 release'})
      result: Marvel's Spider-Man 2 [PlayStation 5, 2023] (id 005, distance 0.1778)
      result: Marvel's Spider-Man [PlayStation 4, 2018] (id 004, distance 0.1926)
      result: Halo Infinite [Xbox Series X|S, 2021] (id 015, distance 0.2007)

[2] TOOL CALL -> evaluate_retrieval({'question': 'Was Mortal Kombat X released for PlayStation 5?', 'retrieved_docs': '[{"id":"005","Name":"Marvel\'s Spider-Man 2","Platform":"PlayStation 5","YearOfRe...'})
      VERDICT: useful=False | co

In [14]:
# Query 4 - extra: publisher question answered from the dataset
ask("Who published Halo Infinite and in which year was it released?", session_id="q4")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: Who published Halo Infinite and in which year was it released?
----------------------------------------------------------------------------------------------------
State machine path: __entry__ -> message_prep -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor -> __termination__

[1] TOOL CALL -> retrieve_game({'query': 'Halo Infinite'})
      result: Halo Infinite [Xbox Series X|S, 2021] (id 015, distance 0.0949)
      result: Minecraft [Xbox One, 2014] (id 014, distance 0.2001)
      result: Marvel's Spider-Man 2 [PlayStation 5, 2023] (id 005, distance 0.2199)

[2] TOOL CALL -> evaluate_retrieval({'question': 'Who published Halo Infinite and in which year was it released?', 'retrieved_docs': '[{"id":"015","Name":"Halo Infinite","Platform":"Xbox Series X|S","YearOfRelease"...'})
      VERDICT: useful=True | confidence=high
      reason : The retrieved do

### Conversation state (memory) demo
The follow-up question below has no game name. The agent resolves "it" from the earlier turn stored in the
session `q2` (short-term memory of the `Agent` class), and the same session also keeps the full message history.

In [15]:
# Follow-up in the SAME session as Query 2 - depends on the previous answer
ask("Which platform was it released on, and who published it?", session_id="q2")

runs = udaplay_agent.get_session_runs("q2")
print(f"\nSession 'q2' holds {len(runs)} runs; user questions remembered:")
for m in runs[-1].get_final_state()["messages"]:
    if isinstance(m, UserMessage):
        print("  -", m.content)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor


[StateMachine] Executing step: tool_executor


[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
QUESTION: Which platform was it released on, and who published it?
----------------------------------------------------------------------------------------------------
State machine path: __entry__ -> message_prep -> llm_processor -> tool_executor -> llm_processor -> tool_executor -> llm_processor -> __termination__

[1] TOOL CALL -> retrieve_game({'query': 'Super Mario 64 platform and publisher'})
      result: Super Mario 64 [Nintendo 64, 1996] (id 009, distance 0.0776)
      result: Super Mario World [Super Nintendo Entertainment System (SNES), 1990] (id 008, distance 0.1153)
      result: Mario Kart 8 Deluxe [Nintendo Switch, 2017] (id 012, distance 0.1584)

[2] TOOL CALL -> evaluate_retrieval({'question': 'Which platform was it released on, and who published it?', 'retrieved_docs': '[{"id":"009","Name":"Super Mario 64","Platform":"Nintendo 64","YearOfRelease":19...'})
      VERDICT: useful=Tru

### Run summary

In [16]:
# Summary of every query run in this notebook (data for the report below)
for i, row in enumerate(trace_log, start=1):
    print(f"{i}. {row['question']}")
    print(f"   path={row['path']} | evaluation useful={row['useful']} | judge confidence={row['confidence']}")
    if row["web_urls"]:
        print(f"   web sources: {row['web_urls'][:3]}")

1. When was Pokémon Gold and Silver released?
   path=internal | evaluation useful=True | judge confidence=high
2. Which one was the first 3D platformer Mario game?
   path=internal | evaluation useful=True | judge confidence=high
3. Was Mortal Kombat X released for PlayStation 5?
   path=web | evaluation useful=False | judge confidence=high
   web sources: ['https://www.youtube.com/watch?v=tqsw711ZuAk', 'https://www.target.com/s/mortal+kombat+ps', 'https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V']
4. Who published Halo Infinite and in which year was it released?
   path=internal | evaluation useful=True | judge confidence=high
5. Which platform was it released on, and who published it?
   path=internal | evaluation useful=True | judge confidence=high


## Report

Every query went through the same state machine (`message_prep → llm_processor ⇄ tool_executor → termination`).
Tool order for each query: `retrieve_game` → `evaluate_retrieval` → `game_web_search` (only when the evaluation said the internal documents were not enough) → final answer.

| # | Question | Path taken | Final answer | Source / citation | Confidence |
|---|----------|-----------|--------------|-------------------|------------|
| 1 | When was Pokémon Gold and Silver released? | Internal (judge: useful) | Released in 1999 on Game Boy Color (publisher: Nintendo). | UdaPlay internal knowledge base, game ID 006 | High |
| 2 | Which one was the first 3D platformer Mario game? | Internal (judge: useful) | Super Mario 64 (Nintendo 64, 1996, Nintendo). | UdaPlay internal knowledge base, game ID 009 | High |
| 3 | Was Mortal Kombat X released for PlayStation 5? | **Web fallback** (judge: not useful, the game is not in the dataset) | No native PS5 release. It came out on 14 April 2015 for PS4 and is playable on PS5 through backward compatibility and the PlayStation Plus Collection. | https://www.mortalkombatonline.com/t/mkx/ps5-playstation-plus-collection-to-include-mortal-kombat-x/85LJ3iL2YW7V | High (web sources agree) |
| 4 | Who published Halo Infinite and in which year was it released? | Internal (judge: useful) | Xbox Game Studios, 2021. | UdaPlay internal knowledge base, game ID 015 | High |
| 5 | Follow-up in the same session as #2: "Which platform was it released on, and who published it?" | Internal (judge: useful) | The agent resolved "it" to Super Mario 64: Nintendo 64, published by Nintendo. | UdaPlay internal knowledge base, game ID 009 | High |

**What worked:** retrieval ranked the right game first for every question that is in the dataset (smallest distance). The LLM judge accepted those documents and correctly rejected the documents retrieved for Mortal Kombat X, so the web fallback ran only for that query. The web answer cites URLs. Session memory let the agent resolve the follow-up question ("it" = Super Mario 64).

**Limits:** the dataset has only 15 games and one platform per game, so multi-platform questions need the web. The workflow order is enforced by the system prompt, not by hard-coded transitions, so a different model could skip a step. Web answers depend on Tavily result quality (some results are YouTube, retail or forum pages), and the judge's yes/no verdict is itself an LLM decision.

### (Optional) Advanced

In [17]:
# (Optional) Advanced - not implemented in this submission:
# - long-term memory (lib.memory.LongTermMemory) to store facts learned from web searches
# - a state machine with the tools as pre-defined nodes